<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/15-unsupervised-learning/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 15: Solutions

Try the exercises first. The notes after each solution explain the idea behind it and the mistake each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math
import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, rand_score, silhouette_samples
from sklearn.mixture import GaussianMixture


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _blobs(seed=0, n=300, centers=4, std=1.0):
    X, y = make_blobs(n_samples=n, centers=centers, cluster_std=std, random_state=seed)
    return X, y


def check_assign_to_nearest(f):
    rng = np.random.default_rng(0)
    X = rng.normal(size=(500, 3))
    C = rng.normal(size=(6, 3))
    reference = ((X[:, None, :] - C[None]) ** 2).sum(axis=2).argmin(axis=1)
    _scenarios("assign_to_nearest", [
        ("the example", lambda: f(np.array([[0, 0], [9, 9], [4, 4]]), np.array([[1, 1], [8, 8]])).tolist(), [0, 1, 0],
         "squared distance from every point to every centroid, then argmin"),
        ("ties go to the lower index", lambda: f(np.array([[0.0, 0.0]]), np.array([[1.0, 0.0], [-1.0, 0.0]])).tolist(), [0], None),
        ("random data in 3 dimensions", lambda: f(X, C).tolist(), reference.tolist(), None),
        ("returns integers", lambda: f(X, C).dtype.kind, "i", None),
    ])


def check_inertia(f):
    X, _ = _blobs(1)
    km = KMeans(n_clusters=4, n_init=5, random_state=0).fit(X)
    _scenarios("inertia", [
        ("a small example", lambda: f(np.array([[0, 0], [2, 0], [10, 10]]), np.array([0, 0, 1]), np.array([[1, 0], [10, 10]])), 2.0,
         "sum of squared distances to each point's own centroid"),
        ("matches scikit-learn's inertia_", lambda: f(X, km.labels_, km.cluster_centers_), float(km.inertia_), None),
        ("zero when every point is a centroid", lambda: f(X[:5], np.arange(5), X[:5]), 0.0, None),
        ("returns a float", lambda: type(f(X, km.labels_, km.cluster_centers_)).__name__, "float", None),
    ])


def check_robust_z_scores(f):
    rng = np.random.default_rng(2)
    x = np.r_[rng.normal(50, 5, 200), 500.0]
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    _scenarios("robust_z_scores", [
        ("a small example", lambda: f([1, 2, 3, 4, 100]), (np.array([1, 2, 3, 4, 100]) - 3) / (1.4826 * 1.0),
         "median 3, MAD = median of |x - 3| = 1"),
        ("matches the formula", lambda: f(x), (x - med) / (1.4826 * mad), None),
        ("the outlier stands out far more than with ordinary z-scores",
         lambda: bool(f(x)[-1] > 3 * ((x[-1] - x.mean()) / x.std())), True, None),
        ("normal data: about the same as ordinary z-scores", lambda: abs(np.std(f(rng.normal(0, 1, 20000))) - 1) < 0.05, True, None),
        ("MAD of 0 raises", lambda: f([5, 5, 5, 5, 9]), Raises(ValueError), "check whether the MAD is 0"),
    ])


def check_cluster_profile(f):
    df = pd.DataFrame({"spend": [10.0, 12, 11, 100, 120, 5000], "visits": [1, 2, 3, 10, 11, 12]}, index=[10, 3, 7, 1, 4, 99])
    labels = [1, 1, 1, 0, 0, 0]
    expected = pd.DataFrame({"size": [3, 3], "share": [0.5, 0.5], "spend": [120.0, 11.0], "visits": [11.0, 2.0]}, index=[0, 1])
    before = df.copy()

    def unchanged():
        f(df, labels)
        return bool(df.equals(before))

    def strip(t):
        out = t.copy()
        out.index = list(out.index)
        out.index.name = None
        return out

    _scenarios("cluster_profile", [
        ("a small example, with a shuffled index", lambda: strip(f(df, labels)), expected,
         "one row per cluster: size, share, then the median of each column"),
        ("sorted by cluster label", lambda: list(f(df, [2, 0, 1, 2, 0, 1]).index), [0, 1, 2], None),
        ("size is an int", lambda: f(df, labels)["size"].dtype.kind, "i", None),
        ("df is not changed", unchanged, True, "don't add a column to df"),
    ])


def check_kmeans(f):
    X, _ = _blobs(3, n=400, centers=5, std=1.5)
    rng = np.random.default_rng(3)
    init = X[rng.choice(len(X), 5, replace=False)]
    sk = KMeans(n_clusters=5, init=init, n_init=1, algorithm="lloyd", max_iter=300, tol=0).fit(X)
    init_copy = init.copy()

    def untouched():
        f(X, init)
        return bool(np.array_equal(init, init_copy))

    def inertia_never_rises():
        values = []
        for steps in range(1, 8):
            labels, centroids, _ = f(X, init, max_iter=steps)
            values.append(((X - centroids[labels]) ** 2).sum())
        return bool(all(b <= a + 1e-9 for a, b in zip(values, values[1:])))

    _scenarios("kmeans", [
        ("a tiny example", lambda: tuple(np.asarray(v).tolist() for v in f(np.array([[0.0], [1.0], [10.0], [11.0]]), np.array([[0.0], [1.0]]))[:2]),
         ([0, 0, 1, 1], [[0.5], [10.5]]), "assign, update, repeat until no label changes"),
        ("same labels as scikit-learn", lambda: adjusted_rand_score(sk.labels_, f(X, init)[0]), 1.0, None),
        ("same centroids as scikit-learn", lambda: np.round(np.sort(f(X, init)[1], axis=0), 6),
         np.round(np.sort(sk.cluster_centers_, axis=0), 6), None),
        ("max_iter is respected", lambda: f(X, init, max_iter=2)[2], 2, None),
        ("an empty cluster keeps its centroid", lambda: f(np.array([[0.0], [1.0]]), np.array([[0.5], [100.0]]))[1].tolist(), [[0.5], [100.0]],
         "skip clusters with no points"),
        ("the inertia never goes up", inertia_never_rises, True, None),
        ("the starting centroids are not changed", untouched, True, "work on a copy"),
    ])


def check_kmeans_plus_plus(f):
    X, y = _blobs(4, n=1000, centers=10, std=0.4)

    def reference(Xr, k, seed):
        g = np.random.default_rng(seed)
        c = [Xr[g.integers(len(Xr))]]
        d2 = ((Xr - c[0]) ** 2).sum(axis=1)
        while len(c) < k:
            nxt = Xr[g.choice(len(Xr), p=d2 / d2.sum())]
            c.append(nxt)
            d2 = np.minimum(d2, ((Xr - nxt) ** 2).sum(axis=1))
        return np.array(c)

    def groups_found(starts):
        return len(set(y[((X[:, None, :] - starts[None]) ** 2).sum(axis=2).argmin(axis=0)]))

    def plus_plus_spreads_out():
        pp = np.mean([groups_found(f(X, 10, seed=s)) for s in range(20)])
        random_starts = np.mean([groups_found(X[np.random.default_rng(s).choice(len(X), 10, replace=False)]) for s in range(20)])
        return bool(pp > random_starts + 1)

    _scenarios("kmeans_plus_plus", [
        ("matches the reference procedure", lambda: f(X, 5, seed=0), reference(X, 5, 0), "one rng: integers for the first, choice with p for the rest"),
        ("the seed is used", lambda: f(X, 3, seed=7), reference(X, 3, 7), None),
        ("returns k distinct points from X", lambda: (f(X, 10).shape, len({tuple(r) for r in f(X, 10)})), ((10, 2), 10), None),
        ("spreads out better than random starts", plus_plus_spreads_out, True, None),
        ("k = 0 raises", lambda: f(X, 0), Raises(ValueError), None),
        ("k larger than n raises", lambda: f(X[:3], 4), Raises(ValueError), None),
    ])


def check_silhouette_samples_manual(f):
    X, _ = _blobs(5, n=200, centers=3, std=1.2)
    labels = KMeans(n_clusters=3, n_init=5, random_state=0).fit_predict(X)
    wrong = labels.copy()
    far_point = int(np.argmax(X[:, 0]))
    wrong[far_point] = (labels[far_point] + 1) % 3
    tight, tight_y = _blobs(6, n=150, centers=[[0, 0], [20, 20], [-20, 20]], std=0.5)
    single = np.r_[np.zeros(5, dtype=int), 1]
    X_single = np.r_[np.random.default_rng(0).normal(size=(5, 2)), [[10.0, 10.0]]]
    _scenarios("silhouette_samples_manual", [
        ("matches scikit-learn", lambda: f(X, labels), silhouette_samples(X, labels), None),
        ("a point in the wrong cluster is negative", lambda: bool(f(X, wrong)[far_point] < 0), True, None),
        ("matches scikit-learn with a misplaced point", lambda: f(X, wrong), silhouette_samples(X, wrong), "divide by max(a, b)"),
        ("well-separated blobs average near 1", lambda: bool(f(tight, tight_y).mean() > 0.9), True, None),
        ("a point alone in its cluster gets 0", lambda: float(f(X_single, single)[-1]), 0.0, "singletons get 0"),
        ("string labels work", lambda: f(X, np.array(["a", "b", "c"])[labels]), silhouette_samples(X, labels), None),
        ("one cluster raises", lambda: f(X, np.zeros(len(X))), Raises(ValueError), None),
    ])


def check_pca_svd(f):
    rng = np.random.default_rng(7)
    X = rng.normal(size=(300, 5)) @ rng.normal(size=(5, 5)) + 100       # correlated, and far from the origin
    sk = PCA(n_components=3).fit(X)
    result = {}

    def run():
        if "v" not in result:
            result["v"] = f(X, 3)
        return result["v"]

    def scores_uncorrelated():
        s = run()["scores"]
        cov = np.cov(s, rowvar=False)
        return bool(np.allclose(cov - np.diag(np.diag(cov)), 0, atol=1e-8) and np.allclose(np.diag(cov), run()["explained_variance"]))

    _scenarios("pca_svd", [
        ("mean", lambda: run()["mean"], X.mean(axis=0), None),
        ("components match scikit-learn", lambda: run()["components"], sk.components_,
         "centre first; flip each component so its largest-magnitude entry is positive"),
        ("explained variance", lambda: run()["explained_variance"], sk.explained_variance_, "S² / (n - 1)"),
        ("explained variance ratio", lambda: run()["explained_variance_ratio"], sk.explained_variance_ratio_,
         "divide by the total variance of all directions"),
        ("scores match transform", lambda: run()["scores"], sk.transform(X), None),
        ("scores are uncorrelated, with the explained variances", scores_uncorrelated, True, None),
    ])


def check_adjusted_rand_index(f):
    rng = np.random.default_rng(8)
    a = rng.integers(0, 5, 500)
    b = np.where(rng.random(500) < 0.7, a, rng.integers(0, 5, 500))
    r1 = rng.integers(0, 10, 1000)
    r2 = rng.integers(0, 10, 1000)
    _scenarios("adjusted_rand_index", [
        ("identical clusterings give 1", lambda: f([0, 0, 1, 1, 2], [0, 0, 1, 1, 2]), 1.0, None),
        ("renamed clusters still give 1", lambda: f([0, 0, 1, 1, 2], ["x", "x", "z", "z", "y"]), 1.0, "only the grouping matters"),
        ("matches scikit-learn", lambda: f(a, b), float(adjusted_rand_score(a, b)), None),
        ("different numbers of clusters", lambda: f(a, a // 2), float(adjusted_rand_score(a, a // 2)), None),
        ("unrelated clusterings: about 0, while the plain Rand index is high",
         lambda: (abs(f(r1, r2)) < 0.02, rand_score(r1, r2) > 0.8), (True, True), None),
        ("everything in one cluster in both gives 1", lambda: f([1, 1, 1], [5, 5, 5]), 1.0, "if the denominator is 0, return 1.0"),
        ("returns a float", lambda: type(f(a, b)).__name__, "float", None),
    ])


def check_dbscan(f):
    rng = np.random.default_rng(9)
    X_m, _ = make_moons(n_samples=300, noise=0.06, random_state=0)
    X = np.vstack([X_m, rng.uniform([-1.5, -1], [2.5, 1.5], size=(30, 2))])
    eps, ms = 0.15, 5
    sk = DBSCAN(eps=eps, min_samples=ms).fit(X)
    core = np.zeros(len(X), dtype=bool)
    core[sk.core_sample_indices_] = True
    D = np.sqrt(((X[:, None, :] - X[None]) ** 2).sum(axis=2))
    result = {}

    def run():
        if "v" not in result:
            result["v"] = np.asarray(f(X, eps, ms))
        return result["v"]

    def borders_ok():
        labels = run()
        for i in np.flatnonzero((labels != -1) & ~core):
            near_core = np.flatnonzero((D[i] <= eps) & core)
            if not np.any(labels[near_core] == labels[i]):
                return False
        return True

    _scenarios("dbscan", [
        ("a small example", lambda: f(np.array([[0.0], [0.1], [0.2], [5.0], [5.1], [5.2], [9.0]]), 0.15, 2).tolist(),
         [0, 0, 0, 1, 1, 1, -1], "neighbourhoods include the point itself"),
        ("a distance exactly eps counts as a neighbour", lambda: f(np.array([[0.0], [1.0], [2.0]]), 1.0, 2).tolist(), [0, 0, 0], "use <= eps"),
        ("the point itself counts towards min_samples", lambda: f(np.array([[0.0], [0.1], [5.0]]), 0.15, 2).tolist(), [0, 0, -1], None),
        ("the same noise points as scikit-learn", lambda: np.flatnonzero(run() == -1).tolist(), np.flatnonzero(sk.labels_ == -1).tolist(), None),
        ("the same clusters of core points (ARI)", lambda: float(adjusted_rand_score(sk.labels_[core], run()[core])), 1.0, None),
        ("clusters numbered from 0 in order of discovery", lambda: sorted(set(run().tolist()) - {-1}), list(range(len(set(sk.labels_) - {-1}))), None),
        ("every border point is next to a core point of its cluster", borders_ok, True, "only explore the neighbourhoods of core points"),
        ("min_samples too high: everything is noise", lambda: set(f(X, 0.05, 50).tolist()), {-1}, None),
    ])


def check_single_linkage_clusters(f):
    X_m, y_m = make_moons(n_samples=200, noise=0.05, random_state=1)
    rng = np.random.default_rng(10)
    X_bridge = np.vstack([X_m, np.column_stack([np.linspace(-0.2, 1.2, 25), np.full(25, 0.25)]) + rng.normal(0, 0.01, (25, 2))])
    X_b, _ = _blobs(11, n=120, centers=3, std=1.0)
    _scenarios("single_linkage_clusters", [
        ("a small example", lambda: f(np.array([[0.0], [0.5], [5.0], [5.4], [20.0]]), 3).tolist(), [0, 0, 1, 1, 2],
         "merge the closest pair of clusters, again and again"),
        ("labels numbered by smallest point index", lambda: f(np.array([[9.0], [0.0], [0.1], [9.1]]), 2).tolist(), [0, 1, 1, 0], None),
        ("matches scikit-learn on blobs", lambda: float(adjusted_rand_score(
            AgglomerativeClustering(n_clusters=3, linkage="single").fit_predict(X_b), f(X_b, 3))), 1.0, None),
        ("separates two clean moons", lambda: float(adjusted_rand_score(y_m, f(X_m, 2))), 1.0, None),
        ("matches scikit-learn when a bridge joins the moons", lambda: float(adjusted_rand_score(
            AgglomerativeClustering(n_clusters=2, linkage="single").fit_predict(X_bridge), f(X_bridge, 2))), 1.0, None),
        ("n_clusters = n gives every point its own cluster", lambda: f(X_b[:6], 6).tolist(), list(range(6)), None),
    ])


def check_gaussian_mixture_1d(f):
    rng = np.random.default_rng(12)
    x = np.r_[rng.normal(-2, 0.7, 600), rng.normal(3, 1.5, 400)]

    def reference(xx, mu, var, w, n_iter):
        mu, var, w = np.array(mu, float), np.array(var, float), np.array(w, float)
        hist = []
        for _ in range(n_iter + 1):
            dens = w * np.exp(-(xx[:, None] - mu) ** 2 / (2 * var)) / np.sqrt(2 * np.pi * var)
            tot = dens.sum(axis=1)
            hist.append(float(np.log(tot).sum()))
            if len(hist) == n_iter + 1:
                break
            r = dens / tot[:, None]
            Nk = r.sum(axis=0)
            w = Nk / len(xx)
            mu = (r * xx[:, None]).sum(axis=0) / Nk
            var = (r * (xx[:, None] - mu) ** 2).sum(axis=0) / Nk
        return {"means": mu, "variances": var, "weights": w, "log_likelihood": hist}

    start = ([-1.0, 1.0], [1.0, 1.0], [0.5, 0.5])
    result = {}

    def run():
        if "v" not in result:
            result["v"] = f(x, *start, n_iter=50)
        return result["v"]

    def close_to_sklearn():
        gm = GaussianMixture(2, means_init=[[-1.0], [1.0]], precisions_init=[[[1.0]], [[1.0]]], weights_init=[0.5, 0.5],
                             reg_covar=0, tol=1e-12, max_iter=500).fit(x[:, None])
        order = np.argsort(gm.means_[:, 0])
        ours = f(x, *start, n_iter=500)
        return bool(np.allclose(ours["means"], gm.means_[order, 0], atol=1e-3) and np.allclose(ours["weights"], gm.weights_[order], atol=1e-3))

    _scenarios("gaussian_mixture_1d", [
        ("one iteration matches the reference", lambda: f(x, *start, n_iter=1), reference(x, *start, 1),
         "E-step: responsibilities; M-step: weights, then means, then variances with the new means"),
        ("fifty iterations match the reference", run, reference(x, *start, 50), None),
        ("n_iter + 1 log-likelihood values", lambda: len(run()["log_likelihood"]), 51, None),
        ("the log-likelihood never decreases", lambda: bool(np.all(np.diff(run()["log_likelihood"]) >= -1e-9)), True, None),
        ("recovers the true means (-2, 3) and weights (0.6, 0.4)",
         lambda: (np.round(run()["means"], 0).tolist(), np.round(run()["weights"], 1).tolist()), ([-2.0, 3.0], [0.6, 0.4]), None),
        ("matches scikit-learn's GaussianMixture from the same start", close_to_sklearn, True, None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `assign_to_nearest`

In [2]:
def assign_to_nearest(X, centroids) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    centroids = np.asarray(centroids, dtype=float)
    squared = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)    # shape (n, k)
    return squared.argmin(axis=1)

In [3]:
check_assign_to_nearest(assign_to_nearest)

PASS  all 4 tests passed for assign_to_nearest


`X[:, None, :] - centroids[None, :, :]` has shape `(n, k, d)`: every point minus every centroid, in one step (broadcasting, Module 6). Summing the squares over the last axis gives an `(n, k)` table of squared distances, and `argmin` along each row picks the nearest centroid, returning the first index on ties.

There's no need for the square root: the nearest centroid by squared distance is the nearest by distance, and skipping `sqrt` saves time.

The `(n, k, d)` array can get big: a million points, 100 centroids and 50 features is 5 billion numbers. scikit-learn avoids this by expanding `|x - c|² = |x|² - 2 x·c + |c|²`, so the middle term is one matrix product, and by working in chunks.

### Exercise 2: `inertia`

In [4]:
def inertia(X, labels, centroids) -> float:
    X = np.asarray(X, dtype=float)
    centroids = np.asarray(centroids, dtype=float)
    return float(((X - centroids[np.asarray(labels)]) ** 2).sum())

In [5]:
check_inertia(inertia)

PASS  all 4 tests passed for inertia


`centroids[labels]` uses **fancy indexing** (Module 6) to build an array with one centroid per point: row `i` is the centroid of point `i`'s cluster. Then it's a subtraction, a square and a sum.

Inertia always falls as `k` grows, reaching 0 when every point is its own cluster, so it can't choose `k` on its own; that's why the elbow method looks for where it **stops falling quickly**.

It's also why k-means is sensitive to scale: a feature measured in large units contributes large squared differences and dominates the total. The check confirms that your value matches scikit-learn's `inertia_`.

### Exercise 3: `robust_z_scores`

In [6]:
def robust_z_scores(x) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    median = np.median(x)
    mad = np.median(np.abs(x - median))
    if mad == 0:
        raise ValueError("the median absolute deviation is 0, so robust z-scores are undefined")
    return (x - median) / (1.4826 * mad)

In [7]:
check_robust_z_scores(robust_z_scores)

PASS  all 5 tests passed for robust_z_scores


The median and the MAD barely move when a few values are extreme, while the mean and the standard deviation are dragged towards them. The check shows the difference: with one huge outlier in the data, the ordinary z-score of that outlier is modest (the outlier itself inflates the standard deviation), while its robust z-score is enormous.

A common rule flags values with `|z| > 3.5` as potential outliers (a recommendation from Boris Iglewicz and David Hoaglin's 1993 guide to outlier detection).

The `MAD == 0` case is real: a column where most values are 0 (say, "number of complaints") has a MAD of 0, and every non-zero value would get an infinite score. That's a sign the method doesn't fit the data, so raising an error is better than returning infinities.

### Exercise 4: `cluster_profile`

In [8]:
def cluster_profile(df: pd.DataFrame, labels) -> pd.DataFrame:
    labels = pd.Series(np.asarray(labels), index=df.index, name="cluster")
    medians = df.groupby(labels).median()
    sizes = labels.value_counts().sort_index()
    profile = pd.DataFrame({"size": sizes.astype(int), "share": sizes / len(labels)})
    return pd.concat([profile, medians], axis=1).sort_index()

In [9]:
check_cluster_profile(cluster_profile)

PASS  all 4 tests passed for cluster_profile


This is the step that turns cluster numbers into something a person can use: how big is each group, and what's typical of it? `groupby` (Module 7) with an external Series of labels does the work without adding a column to `df`, so the caller's DataFrame stays unchanged.

Giving the labels Series the same index as `df` matters: `groupby` aligns on the index, and a DataFrame that has been filtered or sorted won't have the index `0, 1, 2, ...`. The check uses such a DataFrame.

Why medians? Clusters of spending, income or prices usually have skewed values, and a few very large customers would pull the mean far from what's typical. Report the profile in the **original units**, not standardised ones: "median spend on groceries 9,000" means something to a person; "1.3 standard deviations above average" mostly doesn't.

## Medium

### Exercise 5: `kmeans`

In [10]:
def kmeans(X, initial_centroids, max_iter: int = 300):
    X = np.asarray(X, dtype=float)
    centroids = np.array(initial_centroids, dtype=float)       # a copy, so the caller's array isn't changed
    labels = None
    for n_iter in range(1, max_iter + 1):
        new_labels = assign_to_nearest(X, centroids)
        if labels is not None and np.array_equal(new_labels, labels):
            break
        labels = new_labels
        for j in range(len(centroids)):
            members = X[labels == j]
            if len(members) > 0:
                centroids[j] = members.mean(axis=0)
    return labels, centroids, n_iter

In [11]:
check_kmeans(kmeans)

PASS  all 7 tests passed for kmeans


The loop alternates the two steps from the lesson. Stopping when **no label changes** is exact: if the assignments are the same, the centroids computed from them are the same too, so nothing can change any more.

`np.array(initial_centroids, dtype=float)` makes a **copy**. Updating the caller's array in place would be a nasty surprise for them; the check tests that it isn't changed.

**Empty clusters** happen when a centroid starts far from every point, or loses all its points to others. The mean of no points is undefined (NumPy would return `nan` with a warning), so we keep the old centroid. scikit-learn instead moves an empty cluster's centroid to the point farthest from its own centroid, which usually gives a better result.

The check confirms that from the same starting centroids your result matches scikit-learn's `KMeans(init=centroids, n_init=1, algorithm="lloyd")`, and that the inertia never goes up from one iteration to the next.

### Exercise 6: `kmeans_plus_plus`

In [12]:
def kmeans_plus_plus(X, k: int, seed: int = 0) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    if not 1 <= k <= len(X):
        raise ValueError(f"k must be between 1 and {len(X)}, got {k}")
    rng = np.random.default_rng(seed)
    centroids = [X[rng.integers(len(X))]]
    d2 = ((X - centroids[0]) ** 2).sum(axis=1)
    while len(centroids) < k:
        nxt = X[rng.choice(len(X), p=d2 / d2.sum())]
        centroids.append(nxt)
        d2 = np.minimum(d2, ((X - nxt) ** 2).sum(axis=1))      # nearest chosen centroid so far
    return np.array(centroids)

In [13]:
check_kmeans_plus_plus(kmeans_plus_plus)

PASS  all 6 tests passed for kmeans_plus_plus


Choosing each new centroid with probability proportional to `d²` makes far-away points much more likely to be picked, so the starting centroids spread out across the data. A point already chosen has `d² = 0` and can never be chosen again.

Keeping `d2` up to date with `np.minimum` is the efficient part: each new centroid can only bring points **closer** to their nearest centroid, so there's no need to recompute distances to all the earlier ones.

Arthur and Vassilvitskii proved that this start alone gives an inertia, on average, within a factor of about `8(ln k + 2)` of the best possible, before Lloyd's algorithm even runs. In practice it's usually far better than that.

The check runs both random starts and k-means++ starts on data with 10 well-separated groups: random starts regularly put two centroids in one group, while k-means++ starts almost always find all ten.

scikit-learn's version adds a refinement: it tries a few candidates at each step and keeps the one that reduces the total `d²` most ("greedy k-means++"), so its choices won't match yours exactly.

### Exercise 7: `silhouette_samples_manual`

In [14]:
def silhouette_samples_manual(X, labels) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    clusters = np.unique(labels)
    if not 2 <= len(clusters) < len(X):
        raise ValueError("need at least 2 clusters and fewer clusters than points")
    distances = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
    # mean distance from every point to every cluster (including itself, fixed below)
    sums = np.column_stack([distances[:, labels == c].sum(axis=1) for c in clusters])
    sizes = np.array([np.sum(labels == c) for c in clusters])
    own = np.searchsorted(clusters, labels)
    own_size = sizes[own]
    rows = np.arange(len(X))
    with np.errstate(invalid="ignore", divide="ignore"):
        a = sums[rows, own] / (own_size - 1)                  # exclude the point itself (its distance is 0)
    others = sums / sizes
    others[rows, own] = np.inf
    b = others.min(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        s = (b - a) / np.maximum(a, b)
    return np.where(own_size > 1, s, 0.0)

In [15]:
check_silhouette_samples_manual(silhouette_samples_manual)

PASS  all 7 tests passed for silhouette_samples_manual


The full table of pairwise distances makes this straightforward: summing a row over the columns of each cluster gives the total distance to that cluster. For a point's own cluster, the sum includes its distance to itself (0), so dividing by `size - 1` gives the average over the **other** points. Setting the own cluster's entry to infinity before taking the minimum leaves `b` as the nearest other cluster.

Singletons get 0 by convention (scikit-learn does the same), because `a` is undefined for a point with no companions.

The pairwise table has `n²` entries, which is fine for thousands of points and too much for millions. That's why `silhouette_score` has a `sample_size` argument for estimating it from a random sample.

What the check confirms beyond matching scikit-learn: the silhouette of a point stuck in the wrong cluster is negative, and well-separated blobs average close to 1. Notice that the silhouette, like k-means, favours round, compact clusters: a correct clustering of two moons gets a lower silhouette than a wrong one that cuts each moon in half.

### Exercise 8: `pca_svd`

In [16]:
def pca_svd(X, n_components: int) -> dict:
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    centred = X - mean
    U, S, Vt = np.linalg.svd(centred, full_matrices=False)
    variance = S ** 2 / (len(X) - 1)
    components = Vt[:n_components].copy()
    largest = components[np.arange(n_components), np.abs(components).argmax(axis=1)]
    components *= np.sign(largest)[:, None]                  # make the largest entry of each row positive
    return {
        "mean": mean,
        "components": components,
        "explained_variance": variance[:n_components],
        "explained_variance_ratio": variance[:n_components] / variance.sum(),
        "scores": centred @ components.T,
    }

In [17]:
check_pca_svd(pca_svd)

PASS  all 6 tests passed for pca_svd


The SVD writes the centred data as `U S Vᵀ`. The rows of `Vᵀ` are orthogonal unit vectors (the principal directions), ordered by the singular values `S`, and the variance of the data along direction `j` is `S_j² / (n - 1)`. This is how scikit-learn's `PCA` works too. The alternative, finding eigenvectors of the covariance matrix, gives the same answer but is less accurate numerically, since forming `XᵀX` squares the condition number (Module 9).

**Centring is essential.** Without it, the first "component" mostly points from the origin to the middle of the data, which describes where the data is rather than how it varies. The check includes data far from the origin to catch this.

**The sign is arbitrary**: a direction and its opposite describe the same axis. Different libraries, or even different versions of the same library, can return opposite signs. A fixed convention makes results comparable; recent versions of scikit-learn use the same one. Compare components with tests like `np.allclose(np.abs(a), np.abs(b))` when the convention may differ.

The scores are **uncorrelated** with each other and have variances equal to `explained_variance`, which the check confirms. That's the precise sense in which PCA "untangles" correlated features.

### Exercise 9: `adjusted_rand_index`

In [18]:
def adjusted_rand_index(labels_a, labels_b) -> float:
    table = pd.crosstab(np.asarray(labels_a), np.asarray(labels_b)).to_numpy()

    def pairs(m):
        return m * (m - 1) / 2

    n = table.sum()
    index = pairs(table).sum()
    sum_a = pairs(table.sum(axis=1)).sum()
    sum_b = pairs(table.sum(axis=0)).sum()
    expected = sum_a * sum_b / pairs(n)
    maximum = (sum_a + sum_b) / 2
    if maximum == expected:
        return 1.0
    return float((index - expected) / (maximum - expected))

In [19]:
check_adjusted_rand_index(adjusted_rand_index)

PASS  all 7 tests passed for adjusted_rand_index


The idea behind the formula: look at every **pair** of points. `Σ C(n_ij)` counts the pairs that both clusterings put together. Some agreement happens by chance even between random clusterings, and `E` is the expected number of such pairs if the labels were shuffled while keeping the cluster sizes. Subtracting `E` and dividing by the largest possible value minus `E` gives a score that is 1 for identical clusterings (up to renaming) and about 0 for unrelated ones. It can even be slightly negative.

`pd.crosstab` builds the contingency table in one line and works with any label types. Renaming clusters just reorders its rows or columns, which doesn't change any of the sums, which is why the ARI ignores names.

This adjustment for chance is what makes the ARI better than plain accuracy for comparing clusterings: two random clusterings of 1,000 points into 10 clusters agree on most pairs (most pairs are "apart" in both), so the unadjusted Rand index would be around 0.82 for completely unrelated labels. The check shows this.

The ARI was introduced by Lawrence Hubert and Phipps Arabie in 1985.

## Hard

### Exercise 10: `dbscan`

In [20]:
def dbscan(X, eps: float, min_samples: int) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    n = len(X)
    distances = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
    neighbours = [np.flatnonzero(distances[i] <= eps) for i in range(n)]
    is_core = np.array([len(nb) >= min_samples for nb in neighbours])
    labels = np.full(n, -1)
    cluster = 0
    for start in range(n):
        if not is_core[start] or labels[start] != -1:
            continue
        labels[start] = cluster
        to_visit = [start]
        while to_visit:
            point = to_visit.pop()
            for other in neighbours[point]:
                if labels[other] == -1:
                    labels[other] = cluster
                    if is_core[other]:
                        to_visit.append(other)
        cluster += 1
    return labels

In [21]:
check_dbscan(dbscan)

PASS  all 8 tests passed for dbscan


Only core points are ever added to `to_visit`, so only their neighbourhoods are explored. A border point gets a label when a core point reaches it, but it doesn't spread the cluster any further. That's what stops a cluster from leaking across a thin bridge of sparse points.

The result for core points doesn't depend on the order of exploration: two core points within `eps` of each other always end up in the same cluster. Border points are the only ambiguity. A border point within `eps` of core points from **two** different clusters joins whichever reaches it first, so different implementations can disagree about it. The check therefore compares your noise points and core-point clusters with scikit-learn's exactly, and only checks that each border point belongs to a cluster that has a core point within `eps`.

The distance matrix costs `n²` memory. Real implementations find neighbourhoods with spatial indexes (k-d trees or ball trees, the same structures `NearestNeighbors` uses), which makes DBSCAN practical for millions of points in low dimensions.

### Exercise 11: `single_linkage_clusters`

In [22]:
def single_linkage_clusters(X, n_clusters: int) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    n = len(X)
    if not 1 <= n_clusters <= n:
        raise ValueError(f"n_clusters must be between 1 and {n}")
    D = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
    np.fill_diagonal(D, np.inf)
    members = {i: [i] for i in range(n)}       # cluster id -> its points
    active = np.ones(n, dtype=bool)
    for _ in range(n - n_clusters):
        flat = np.argmin(np.where(active[:, None] & active[None, :], D, np.inf))
        i, j = divmod(int(flat), n)
        # merge j into i; single linkage: the distance to the merged cluster is the smaller of the two
        D[i, :] = np.minimum(D[i, :], D[j, :])
        D[:, i] = D[i, :]
        D[i, i] = np.inf
        active[j] = False
        members[i].extend(members.pop(j))
    labels = np.empty(n, dtype=int)
    for new_id, points in enumerate(sorted(members.values(), key=min)):
        labels[points] = new_id
    return labels

In [23]:
check_single_linkage_clusters(single_linkage_clusters)

PASS  all 6 tests passed for single_linkage_clusters


The key fact that makes this efficient is the **update rule** for single linkage: after merging clusters `i` and `j`, the distance from the merged cluster to any other cluster `k` is `min(D[i, k], D[j, k])`, because the closest pair between the merged cluster and `k` is the closest pair from either part. So each merge updates one row and one column of the distance matrix instead of recomputing everything. Complete linkage uses `max` instead, and average and Ward linkage have their own update formulas (together they're known as the Lance-Williams formulas).

Inactive clusters are hidden by setting their distances to infinity before `argmin`, so they're never chosen again.

This version takes about `n²` work per merge, so `n³` in total: fine for hundreds of points. Single linkage has a much faster special method: it's equivalent to building a **minimum spanning tree** of the points and cutting its longest edges, which is what SciPy and scikit-learn do.

The check compares your clusters with scikit-learn's `AgglomerativeClustering(linkage="single")` using the ARI, and shows single linkage's signature behaviour: it follows chains, so it separates two moons perfectly when there's a clean gap, and lumps everything into one cluster once a few noise points form a bridge.

### Exercise 12: `gaussian_mixture_1d`

In [24]:
def gaussian_mixture_1d(x, means, variances, weights, n_iter: int) -> dict:
    x = np.asarray(x, dtype=float)
    mu = np.array(means, dtype=float)
    var = np.array(variances, dtype=float)
    w = np.array(weights, dtype=float)

    def weighted_densities():
        # shape (n, K): w_k * N(x_i | mu_k, var_k)
        return w * np.exp(-(x[:, None] - mu) ** 2 / (2 * var)) / np.sqrt(2 * np.pi * var)

    history = []
    for _ in range(n_iter):
        dens = weighted_densities()
        total = dens.sum(axis=1)
        history.append(float(np.log(total).sum()))
        r = dens / total[:, None]                              # E-step: responsibilities
        Nk = r.sum(axis=0)                                     # M-step
        w = Nk / len(x)
        mu = (r * x[:, None]).sum(axis=0) / Nk
        var = (r * (x[:, None] - mu) ** 2).sum(axis=0) / Nk
    history.append(float(np.log(weighted_densities().sum(axis=1)).sum()))
    return {"means": mu, "variances": var, "weights": w, "log_likelihood": history}

In [25]:
check_gaussian_mixture_1d(gaussian_mixture_1d)

PASS  all 6 tests passed for gaussian_mixture_1d


EM is k-means with soft assignments. The E-step is the "assign" step, except each point is shared out between the components in proportion to how likely each one is to have produced it. The M-step is the "update" step: each component's new mean is a **weighted** average, with the responsibilities as weights, and likewise for the variance and the share.

The guarantee that makes EM work: **the log-likelihood never decreases** from one iteration to the next. The check tests this. Like k-means, though, EM finds a local maximum that depends on where it starts, which is why scikit-learn's `GaussianMixture` has `n_init`.

Two practical dangers, both handled in real implementations:

- **Collapse.** If a component's mean lands exactly on one data point and its variance shrinks towards 0, that point's density goes to infinity and so does the likelihood. scikit-learn adds a small `reg_covar` to every variance to prevent this.
- **Underflow.** For points far from every component, all the densities can round to 0, and `log(0)` is `-inf`. Real implementations work with log-densities and the log-sum-exp trick (Module 9's `log_sum_exp` exercise).

With two components starting near the right places, a few dozen iterations recover the true parameters of the check's data closely, and the result matches scikit-learn's `GaussianMixture` started from the same values.

## What's next

The mini project in `04_mini_project.ipynb` uses these tools on a real wholesale distributor's customers: spending patterns explored with PCA, unusual customers flagged, the number of groups chosen with several tools and tested for stability, and the groups described and checked against what the business already knows.